### 1: Imports


In [ ]:
import os
import re
import pickle
import random
import numpy as np
import pandas as pd
import pathlib as pl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageEnhance, ImageFilter
import einops as eo
import tqdm.auto as tqdm

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

print("All imports successful")

### 2: Set Globals


In [ ]:
# Device configuration
device = torch.device("mps" if torch.backends.mps.is_built() else "cpu")
print(f"Using device: {device}")

# Model configuration - must match the trained model
SIDE_LENGTH = 320
CODE_SIZE = 120
NOISE_RATE = 0  # No noise during inference

target_size = (SIDE_LENGTH, SIDE_LENGTH)
BATCH_SIZE = 32

# Visualisation configuration
N_VIS_SAMPLE = 4  # Number of images to show in visualisation

# Clustering configuration
CLUSTERING_METHOD = 'kmeans'  # Options: 'hierarchical', 'kmeans', 'gmm', 'dbscan'
N_FEATURES_FOR_CLUSTERING = 2       # Set to None to use full latent space (CODE_SIZE dimensions)

print(f"Side length: {SIDE_LENGTH}")
print(f"Code size: {CODE_SIZE}")
print(f"Target size: {target_size}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Clustering method: {CLUSTERING_METHOD}")
print(f"Features for clustering: {N_FEATURES_FOR_CLUSTERING}")

### Set metadata variables:

In [ ]:
analysis_type = 'clustering'
model_name = 'apply_clustering'
python_script = 'img_to_pytorch_apply.ipynb'
notes = 'trained variational convolutional autoencoder, applied to new data (giub data) without retraining'
autoencoder_implementation = 'Custom PyTorch VariationalConvolutionalAutoencoder'
auto_encoder_name = 'variational convolutional autoencoder'

### 3: Define File Paths


In [ ]:
# Project paths
project_path = Path.cwd()
root_path = (project_path / '..' / 'data_folders').resolve()

# Folder containing the images
image_dir = root_path / 'data_jpg'

# Folder containing the labels.csv (image_ids and labels, no file_paths yet)
labels_dir = root_path / 'data'

# Path to the trained model checkpoint (.pth file)
model_checkpoint_path = root_path / 'combined_data' / 'vae_model_20260314_211803' / 'model_059.pth'

# File source label for this dataset (user defined)
FILE_SOURCE = 'giub'

# Path to save results
results_dir = labels_dir

print(f"Image directory: {image_dir}")
print(f"Labels directory: {labels_dir}")
print(f"Model checkpoint: {model_checkpoint_path}")
print(f"File source: {FILE_SOURCE}")
print(f"Results directory: {results_dir}")
print(f"\nImage directory exists: {image_dir.exists()}")
print(f"Labels file exists: {(labels_dir / 'labels_new.csv').exists()}")
print(f"Model checkpoint exists: {model_checkpoint_path.exists()}")

### 4: Load labels.csv and Add File Paths and File Source


In [ ]:
# Load labels.csv (contains image_ids and labels, no file_paths yet)
labels_path = labels_dir / 'labels_new.csv'
labels_df = pd.read_csv(labels_path)

print(f"Loaded labels_new.csv: {labels_df.shape[0]} rows")
print(f"Columns: {list(labels_df.columns)}")
print(labels_df.head())

# Get all image files in image_dir
image_files = [f for f in os.listdir(image_dir) if f.lower().endswith('.jpg')]
print(f"\nFound {len(image_files)} jpg files in image directory")

# Build a dict mapping trailing integer -> full file path
id_to_filepath = {}
for filename in image_files:
    match = re.search(r'(\d+)(?=\.\w+$)', filename)
    if match:
        image_id = int(match.group(1))
        id_to_filepath[image_id] = str(image_dir / filename)
    else:
        print(f"Warning: could not extract integer from filename: {filename}")

print(f"Successfully mapped {len(id_to_filepath)} filenames to image_ids")

# Match image_ids in labels_df to file paths
labels_df['file_paths'] = labels_df['image_id'].map(id_to_filepath)
labels_df['file_source'] = FILE_SOURCE

# Check for unmatched ids
n_missing = labels_df['file_paths'].isna().sum()
if n_missing > 0:
    print(f"\nWarning: {n_missing} image_ids could not be matched to a file")
    print(labels_df[labels_df['file_paths'].isna()])
else:
    print(f"\nAll image_ids successfully matched to file paths")

print(labels_df.head())

In [ ]:
# Save labels file with file paths and file source:
#labels_new_csv_path = labels_dir/'labels_new.csv'
#labels_df.to_csv(labels_new_csv_path)

### 5. Detect Label Columns and Load Image Paths and Labels

In [ ]:
# Auto-detect label columns (everything except image_id, file_paths, file_source)
non_label_cols = {'image_id', 'file_paths', 'file_source'}
label_columns = [col for col in labels_df.columns if col not in non_label_cols]

print(f"Detected label columns: {label_columns}")

# Load image paths
image_paths = list(labels_df['file_paths'])
print(f"\nTotal images: {len(image_paths)}")

# Load labels for each label column
labels_dict = {col: list(labels_df[col]) for col in label_columns}
for col, vals in labels_dict.items():
    print(f"Label '{col}': {len(vals)} values, unique values: {sorted(set(vals))}")

# Verify all files exist on disk
missing = [p for p in image_paths if not os.path.exists(p)]
if missing:
    print(f"\nWarning: {len(missing)} files not found on disk:")
    for p in missing:
        print(f"  {p}")
else:
    print(f"\nAll {len(image_paths)} files verified on disk")

### 6: Function Definitions — Data Loading Pipeline

In [ ]:
def load_and_preprocess_image(image_path, target_size=(320, 320), convert_grayscale=True):
    try:
        image = Image.open(image_path)
        if convert_grayscale:
            image = image.convert('L')
        image = image.resize(target_size, Image.Resampling.LANCZOS)
        return image
    except Exception as e:
        print(f"Error processing {image_path}: {str(e)}")
        return None


def create_transforms(mean=0.5, std=1.0):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((mean,), (std,))
    ])
    return transform


def create_label_transform():
    return transforms.Compose([
        lambda x: torch.LongTensor([x])
    ])


class CustomImageDataset(Dataset):
    def __init__(self, image_paths, labels, target_size=(320, 320),
                 convert_grayscale=True, transform=None, target_transform=None):
        self.image_paths = image_paths
        self.target_size = target_size
        self.convert_grayscale = convert_grayscale
        self.transform = transform
        self.target_transform = target_transform

        if len(image_paths) != len(labels):
            raise ValueError(f"Number of images ({len(image_paths)}) must match number of labels ({len(labels)})")

        self.targets = torch.tensor(labels, dtype=torch.long)
        self._data = None
        print(f"Initializing dataset with {len(image_paths)} images...")

    @property
    def data(self):
        if self._data is None:
            print("Loading all images into .data tensor...")
            self._load_all_images()
        return self._data

    def _load_all_images(self):
        all_images = []
        for i, image_path in enumerate(self.image_paths):
            image = load_and_preprocess_image(image_path, self.target_size, self.convert_grayscale)
            if image is None:
                image = Image.new('L', self.target_size, 0)
            all_images.append(np.array(image, dtype=np.uint8))
            if (i + 1) % 100 == 0:
                print(f"Loaded {i + 1}/{len(self.image_paths)} images...")
        all_images = np.stack(all_images, axis=0)
        self._data = torch.from_numpy(all_images)
        print(f"Loaded .data tensor with shape: {self._data.shape}, dtype: {self._data.dtype}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        raw_image = self.data[idx]
        image = Image.fromarray(raw_image.numpy(), mode='L')
        label = self.targets[idx].item()
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label


def create_image_dataset(image_paths, labels, target_size=(320, 320),
                         convert_grayscale=True, mean=0.5, std=1.0):
    transform = create_transforms(mean=mean, std=std)
    target_transform = create_label_transform()
    dataset = CustomImageDataset(
        image_paths=image_paths,
        labels=labels,
        target_size=target_size,
        convert_grayscale=convert_grayscale,
        transform=transform,
        target_transform=target_transform
    )
    return dataset


print("Data loading functions defined successfully")

### 7: Create Dataset and DataLoader

In [ ]:
# Use the first label column as the primary label for the dataset
# (required by CustomImageDataset, but all labels are available in labels_dict)
primary_label_col = label_columns[5]
print(f"Using '{primary_label_col}' as primary label column")

# Create dataset (no train/val split - apply to all images)
dataset = create_image_dataset(
    image_paths=image_paths,
    labels=labels_dict[primary_label_col],
    target_size=target_size,
    convert_grayscale=True,
    mean=0.5,
    std=1.0
)

# Create DataLoader
data_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # No shuffling - we need to keep track of image order
    drop_last=False  # Keep all images, even if last batch is smaller
)

print(f"\nDataset size: {len(dataset)} images")
print(f"Number of batches: {len(data_loader)}")
print(f"Batch size: {BATCH_SIZE}")

# Quick sanity check on first sample
sample_image, sample_label = dataset[0]
print(f"\nFirst sample:")
print(f"  Image tensor shape: {sample_image.shape}")
print(f"  Image tensor dtype: {sample_image.dtype}")
print(f"  Label: {sample_label}")

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, input_size, code_size):
        self.input_size = list(input_size)
        self.flat_data_size = np.prod(self.input_size)
        self.hidden_size = 128
        self.code_size = code_size
        super(AutoEncoder, self).__init__()

        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flat_data_size, self.hidden_size),
            nn.ReLU(),
            nn.Linear(self.hidden_size, self.code_size),
            nn.Sigmoid(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(self.code_size, self.hidden_size),
            nn.ReLU(),
            nn.Linear(self.hidden_size, self.flat_data_size),
            nn.Tanh(),
            nn.Unflatten(1, self.input_size),
        )

    def forward(self, x, return_z=False):
        encoded = self.encode(x)
        decoded = self.decode(encoded)
        return (decoded, encoded) if return_z else decoded

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z) * 1.1

    def get_n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class VariationalConvolutionalAutoencoder(AutoEncoder):
    def __init__(self, input_size, code_size):
        super(VariationalConvolutionalAutoencoder, self).__init__(input_size, code_size)

        self.input_size = list(input_size)
        self.npix = np.prod(self.input_size)
        self.hidden_size = 64 * 5 * 5  # = 1600
        self.code_size = code_size

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1, stride=1), nn.LeakyReLU(negative_slope=0.3),
            nn.Conv2d(8, 8, 3, padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),
            nn.Conv2d(8, 16, 3, padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),
            nn.Conv2d(16, 16, 3, padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),
            nn.Conv2d(16, 32, 3, padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),
            nn.Conv2d(32, 32, 3, padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),
            nn.Conv2d(32, 64, 3, padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),
            nn.Flatten(),
            nn.Linear(1600, 200), nn.LeakyReLU(negative_slope=0.3),
            nn.Linear(200, self.code_size * 2),
        )

        self.decoder = nn.Sequential(
            nn.Linear(self.code_size, 200), nn.LeakyReLU(negative_slope=0.3),
            nn.Linear(200, 1600), nn.LeakyReLU(negative_slope=0.3),
            nn.Unflatten(1, (64, 5, 5)),
            nn.ConvTranspose2d(64, 32, 3, padding=1, output_padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),
            nn.ConvTranspose2d(32, 32, 3, padding=1, output_padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),
            nn.ConvTranspose2d(32, 16, 3, padding=1, output_padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),
            nn.ConvTranspose2d(16, 16, 3, padding=1, output_padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),
            nn.ConvTranspose2d(16, 8, 3, padding=1, output_padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),
            nn.ConvTranspose2d(8, 8, 3, padding=1, output_padding=1, stride=2), nn.LeakyReLU(negative_slope=0.3),
            nn.Conv2d(8, 1, 3, padding=1, stride=1), nn.Tanh(),
        )

    def encode(self, x):
        z = self.encoder(x)
        z_mean, z_logvar = torch.split(z, self.code_size, dim=1)
        return z_mean, z_logvar

    def reparameterize(self, z_mean, z_logvar):
        eps = torch.randn_like(z_mean)
        z_std = torch.exp(z_logvar * 0.5)
        return eps * z_std + z_mean

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x, return_z=False):
        z_mean, z_logvar = self.encode(x)
        z = self.reparameterize(z_mean, z_logvar)
        reconstruction = self.decode(z)
        if return_z:
            return reconstruction, z_mean, z_logvar
        else:
            return reconstruction

    def forward_and_KL_loss(self, x, y):
        reconstruction, z_mean, z_logvar = self(x, return_z=True)
        loss_z_kl = 0.5 * torch.sum(
            torch.exp(z_logvar) + torch.square(z_mean) - 1.0 - z_logvar,
            dim=1
        )
        loss_z_kl = torch.mean(loss_z_kl) / self.npix
        return reconstruction, loss_z_kl

    def sample(self, eps=None):
        if eps is None:
            eps = torch.randn((100, self.code_size))
        return self.decode(eps)


# Instantiate model
model = VariationalConvolutionalAutoencoder(input_size=target_size, code_size=CODE_SIZE).to(device)
print(f"VAE model instantiated")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

### 9: Load Trained Model from Checkpoint

In [ ]:
# Load checkpoint
checkpoint = torch.load(model_checkpoint_path, weights_only=False)

# Load weights into model
model.load_state_dict(checkpoint['model_state_dict'])

# Set model to evaluation mode (disables dropout etc.)
model.eval()

print(f"Model loaded successfully from: {model_checkpoint_path}")
print(f"Checkpoint was saved at epoch: {checkpoint['epoch']}")
print(f"Training loss at checkpoint: {checkpoint['loss']:.4f}")
print(f"Validation loss at checkpoint: {checkpoint['val_loss']:.4f}")

### 10: Run Inference on All Images — Extract z_mean

In [ ]:
timestamp_start = pd.Timestamp.now()

all_z_mean = []
all_reconstructions = []
all_labels = {col: [] for col in label_columns}

model.eval()
with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(tqdm.tqdm(data_loader)):
        images = images.to(device)
        
        # Forward pass - get reconstruction and latent distribution parameters
        reconstruction, z_mean, z_logvar = model(images, return_z=True)
        
        all_z_mean.append(z_mean.detach().cpu().numpy())
        all_reconstructions.append(reconstruction.detach().cpu().numpy())

# Concatenate results across all batches
all_z_mean = np.concatenate(all_z_mean, axis=0)
all_reconstructions = np.concatenate(all_reconstructions, axis=0)

print(f"Inference complete")
print(f"z_mean shape: {all_z_mean.shape}")
print(f"Reconstructions shape: {all_reconstructions.shape}")
print(f"Expected: {len(image_paths)} images, {CODE_SIZE} latent dimensions")
timestamp_end = pd.Timestamp.now()

### 11: Visualize a Subsample of Original Images vs Reconstructions

In [ ]:
# Randomly select N_VIS_SAMPLE indices for visualisation
vis_indices = random.sample(range(len(dataset)), N_VIS_SAMPLE)

fig, axes = plt.subplots(2, N_VIS_SAMPLE, figsize=(N_VIS_SAMPLE * 4, 8))

for col, idx in enumerate(vis_indices):
    # Original image (from dataset, already preprocessed)
    original = dataset.data[idx].numpy()
    axes[0, col].imshow(original, cmap='gray')
    axes[0, col].set_title(f'Original\n{os.path.basename(image_paths[idx])}', fontsize=8)
    axes[0, col].axis('off')

    # Reconstruction (from inference)
    reconstruction = all_reconstructions[idx][0]  # [0] to remove channel dimension
    axes[1, col].imshow(reconstruction, cmap='gray', vmin=-0.5, vmax=0.5)
    axes[1, col].set_title(f'Reconstruction', fontsize=8)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Reconstruction', fontsize=12)

plt.suptitle('Original vs Reconstructed Images', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Showing {N_VIS_SAMPLE} randomly selected images")

### 12: Optional Dimensionality Reduction

In [ ]:

# Use z_mean as the latent representation for clustering
z_mean = all_z_mean
print(f"Latent space shape: {z_mean.shape}")

if N_FEATURES_FOR_CLUSTERING is None or N_FEATURES_FOR_CLUSTERING >= CODE_SIZE:
    z_reduced = z_mean
    print(f"No dimensionality reduction applied, using full latent space: {z_reduced.shape[1]} dimensions")
else:
    pca = PCA(n_components=N_FEATURES_FOR_CLUSTERING)
    z_reduced = pca.fit_transform(z_mean)
    print(f"PCA applied: {z_mean.shape[1]} → {z_reduced.shape[1]} dimensions")
    print(f"Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

print(f"z_reduced shape: {z_reduced.shape}")

### 13: Clustering Diagnostic Plots

In [ ]:
if CLUSTERING_METHOD == 'hierarchical':
    print("="*80)
    print("HIERARCHICAL CLUSTERING")
    print("="*80)
    
    linkage_matrix = linkage(z_reduced, method='ward')
    
    plt.figure(figsize=(15, 7))
    dendrogram(linkage_matrix, truncate_mode='lastp', p=30)
    plt.title('Hierarchical Clustering Dendrogram', fontsize=16)
    plt.xlabel('Sample Index (or Cluster Size)')
    plt.ylabel('Distance')
    plt.show()
    
    print(f"\n→ Adjust N_CLUSTERS based on dendrogram, then run next cell")

elif CLUSTERING_METHOD == 'kmeans':
    print("="*80)
    print("K-MEANS CLUSTERING")
    print("="*80)
    
    inertias = []
    silhouette_scores = []
    K_range = range(2, 16)
    
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels_k = kmeans.fit_predict(z_reduced)
        inertias.append(kmeans.inertia_)
        silhouette_scores.append(silhouette_score(z_reduced, labels_k))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(K_range, inertias, 'bo-')
    axes[0].set_xlabel('Number of clusters (k)')
    axes[0].set_ylabel('Inertia')
    axes[0].set_title('Elbow Method')
    axes[0].grid(True)
    
    axes[1].plot(K_range, silhouette_scores, 'go-')
    axes[1].set_xlabel('Number of clusters (k)')
    axes[1].set_ylabel('Silhouette Score')
    axes[1].set_title('Silhouette Analysis')
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n→ Adjust N_CLUSTERS based on elbow/silhouette, then run next cell")

elif CLUSTERING_METHOD == 'gmm':
    print("="*80)
    print("GAUSSIAN MIXTURE MODEL (GMM)")
    print("="*80)
    
    bic_scores = []
    aic_scores = []
    K_range = range(2, 16)
    
    for k in K_range:
        gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
        gmm.fit(z_reduced)
        bic_scores.append(gmm.bic(z_reduced))
        aic_scores.append(gmm.aic(z_reduced))
    
    plt.figure(figsize=(10, 5))
    plt.plot(K_range, bic_scores, 'ro-', label='BIC')
    plt.plot(K_range, aic_scores, 'bo-', label='AIC')
    plt.xlabel('Number of components (k)')
    plt.ylabel('Information Criterion')
    plt.title('GMM Model Selection (lower is better)')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    print(f"\n→ Choose N_CLUSTERS where BIC/AIC is lowest, then run next cell")

elif CLUSTERING_METHOD == 'dbscan':
    print("="*80)
    print("DBSCAN (Density-Based Clustering)")
    print("="*80)
    print("Note: DBSCAN automatically determines number of clusters")
    print("You need to set eps (neighborhood radius) and min_samples")
    print("\n→ Run next cell with default parameters, adjust if needed")

else:
    raise ValueError(f"Unknown clustering method: {CLUSTERING_METHOD}")

### 14: Set Number of Clusters

In [ ]:
# ============================================================================
# Adjust N_CLUSTERS based on diagnostic plots above
# ============================================================================

N_CLUSTERS = 4  # Adjust based on dendrogram/elbow/silhouette plots

print(f"Number of clusters: {N_CLUSTERS}")

### 15: Apply Clustering

In [ ]:
if CLUSTERING_METHOD == 'hierarchical':
    cluster_labels = fcluster(linkage_matrix, N_CLUSTERS, criterion='maxclust')
    cluster_labels = cluster_labels - 1  # Convert to 0-indexed
    print(f"✓ Hierarchical clustering complete: {N_CLUSTERS} clusters")

    clustering_implementation = 'scipy.cluster.hierarchy'

elif CLUSTERING_METHOD == 'kmeans':
    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
    cluster_labels = kmeans.fit_predict(z_reduced)
    print(f"✓ K-means clustering complete: {N_CLUSTERS} clusters")

    clustering_implementation = 'sklearn.cluster'

elif CLUSTERING_METHOD == 'gmm':
    gmm = GaussianMixture(n_components=N_CLUSTERS, covariance_type='full', random_state=42)
    cluster_labels = gmm.fit_predict(z_reduced)
    print(f"✓ GMM clustering complete: {N_CLUSTERS} clusters")

    clustering_implementation = 'sklearn.mixture'


    
elif CLUSTERING_METHOD == 'dbscan':
    eps = 3.0
    min_samples = 5
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    cluster_labels = dbscan.fit_predict(z_reduced)
    n_clusters_found = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)
    print(f"✓ DBSCAN complete:")
    print(f"  Clusters found: {n_clusters_found}")
    print(f"  Noise points: {n_noise}")
    print(f"  (eps={eps}, min_samples={min_samples})")

    clustering_implementation = 'sklearn.cluster'

    

# Print cluster distribution
print(f"\nCluster distribution:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Cluster {label}: {count} images")

### 16: t-SNE Visualisation

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
z_2d = tsne.fit_transform(z_reduced)

plt.figure(figsize=(12, 12))
scatter = plt.scatter(z_2d[:, 0], z_2d[:, 1], c=cluster_labels,
                      cmap='tab10', s=100, alpha=0.7, edgecolors='black', linewidth=0.5)
plt.colorbar(scatter, label='Cluster')
plt.title(f'{CLUSTERING_METHOD.upper()} Clustering Results (t-SNE visualization)\n{len(set(cluster_labels))} clusters',
          fontsize=16)
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(True, alpha=0.3)
plt.show()

print(f"t-SNE visualisation complete")

### 17: Build cluster_data DataFrame

In [ ]:
def extract_number(filepath):
    match = re.search(r'(\d+)\.jpg$', filepath)
    if not match:
        raise ValueError(f"No number found at end of filename: {filepath}")
    return int(match.group(1))

# Build base cluster_data dataframe
cluster_data = pd.DataFrame({
    'image_index': list(range(len(cluster_labels))),
    'cluster_label': cluster_labels,
    'image_paths': image_paths,
})

# Extract image_id from file paths
cluster_data['image_id'] = cluster_data['image_paths'].apply(extract_number)

# Add features used in clustering
for i in range(z_reduced.shape[1]):
    cluster_data[f'feature_{i}'] = z_reduced[:, i]

# Join labels from labels_df using image_id
cluster_data = cluster_data.merge(
    labels_df[['image_id'] + label_columns + ['file_source']],
    on='image_id',
    how='left'
)

print(f"cluster_data shape: {cluster_data.shape}")
print(f"Columns: {list(cluster_data.columns)}")
print(cluster_data.head())

### 18: Verify Cluster Data and Print Cluster Summary

In [ ]:
# Check for any unmatched rows after merge
n_missing = cluster_data[label_columns].isna().any(axis=1).sum()
if n_missing > 0:
    print(f"Warning: {n_missing} rows could not be matched to labels")
    print(cluster_data[cluster_data[label_columns].isna().any(axis=1)])
else:
    print(f"✓ All rows successfully matched to labels")

# Print summary per cluster
print(f"\nCluster summary:")
for cluster_id in sorted(set(cluster_labels)):
    cluster_subset = cluster_data[cluster_data.cluster_label == cluster_id]
    print(f"\n  Cluster {cluster_id}: {len(cluster_subset)} images")
    for col in label_columns:
        unique_vals = sorted(set(cluster_subset[col]))
        print(f"    {col}: {unique_vals}")
    print(f"    file_source: {sorted(set(cluster_subset.file_source))}")
    print(f"    Sample images:")
    for path in cluster_subset.image_paths.iloc[:3]:
        print(f"      {os.path.basename(path)}")

### 19: Save Results

In [ ]:
# Generate timestamp_id
timestamp_id = timestamp_start.strftime('%Y%m%d_%H%M%S')

# Initialize metadata
metadata_results = {}
metadata_results['analysis_type'] = analysis_type
metadata_results['model_name'] = model_name
metadata_results['python_script'] = python_script
metadata_results['notes'] = notes
metadata_results['autoencoder_name'] = auto_encoder_name
metadata_results['autoencoder_implementation'] = autoencoder_implementation
metadata_results['autoencoder_params'] = {
    'code_size': CODE_SIZE,
    'noise_rate': NOISE_RATE,
    'side_length': SIDE_LENGTH,
    'in_size': target_size,
    'checkpoint': str(model_checkpoint_path)
}
metadata_results['dim_reduction_name'] = 'PCA' if (N_FEATURES_FOR_CLUSTERING is not None and N_FEATURES_FOR_CLUSTERING < CODE_SIZE) else None
metadata_results['dim_reduction_implementation'] = 'sklearn.decomposition' if (N_FEATURES_FOR_CLUSTERING is not None and N_FEATURES_FOR_CLUSTERING < CODE_SIZE) else None
metadata_results['dim_reduction_params'] = {'n_dimensions': N_FEATURES_FOR_CLUSTERING}
metadata_results['clustering_name'] = CLUSTERING_METHOD
metadata_results['clustering_implementation'] = clustering_implementation
metadata_results['clustering_params'] = {'n_clusters': N_CLUSTERS}
metadata_results['run_timestamp'] = timestamp_id
metadata_results['cluster_data'] = cluster_data

# Save results as pickle
filename = f'results_clustering_{timestamp_id}.pkl'
file_path = results_dir / filename
with open(file_path, 'wb') as f:
    pickle.dump(metadata_results, f)
print(f"Results saved to: {file_path}")

# Verify by reloading
with open(file_path, 'rb') as f:
    reloaded = pickle.load(f)
print(f"Reloaded successfully")
print(f"Keys: {list(reloaded.keys())}")
print(f"cluster_data shape: {reloaded['cluster_data'].shape}")

### 20: Save Times Data

In [ ]:
duration = timestamp_end - timestamp_start
total_seconds = duration.total_seconds()

time_analyses_clustering_pipeline = {
    'analysis_name': ['clustering_pipeline'],
    'time_stamp_start': [timestamp_start],
    'duration_str': [f"Analysis took: {duration}"],
    'duration_seconds': [total_seconds],
    'duration_seconds_str': [f"Analysis took: {total_seconds:.2f} seconds"],
    'duration_minutes': [total_seconds / 60],
    'duration_minutes_str': [f"Analysis took: {total_seconds / 60:.2f} minutes"]
}

time_filename = f'times_clustering_{timestamp_id}.pkl'
time_file_path = results_dir / time_filename
with open(time_file_path, 'wb') as f:
    pickle.dump(time_analyses_clustering_pipeline, f)
print(f"Times data saved to: {time_file_path}")
print(f"Duration: {total_seconds:.2f} seconds ({total_seconds/60:.2f} minutes)")

In [ ]:
def plot_bar_chart(data_dict, title="Bar Chart", xlabel="Keys", ylabel="Values", figsize=(12, 6), 
                   title_fontsize=16, label_fontsize=14, tick_fontsize=12):
    sorted_items = sorted(data_dict.items())
    keys = [item[0] for item in sorted_items]
    values = [item[1] for item in sorted_items]
    
    plt.figure(figsize=figsize)
    plt.bar(range(len(keys)), values)
    plt.title(title, fontsize=title_fontsize)
    plt.xlabel(xlabel, fontsize=label_fontsize)
    plt.ylabel(ylabel, fontsize=label_fontsize)
    plt.grid(axis='y', alpha=0.3)
    plt.xticks(range(len(keys)), keys, fontsize=tick_fontsize)
    plt.yticks(fontsize=tick_fontsize)
    if len(keys) > 10:
        plt.xticks(range(len(keys)), keys, rotation=45, fontsize=tick_fontsize)
    plt.tight_layout()
    plt.show()

print("plot_bar_chart function defined")

In [ ]:
def calculate_label_proportion(cluster_subset, label_name):
    label_sum = sum(cluster_subset[label_name])
    label_proportion = label_sum / cluster_subset.shape[0]
    return label_proportion

for label_col in label_columns:
    # Skip columns with NaN values
    if cluster_data[label_col].isna().any():
        print(f"Skipping '{label_col}' — contains NaN values")
        continue
    
    label_proportions_in_clusters = {}
    for cluster_label in sorted(set(cluster_data.cluster_label)):
        cluster_subset = cluster_data[cluster_data.cluster_label == cluster_label]
        proportion = calculate_label_proportion(cluster_subset, label_col)
        label_proportions_in_clusters[cluster_label] = proportion
    
    plot_bar_chart(
        label_proportions_in_clusters,
        title=f"Proportion of '{label_col}' per cluster",
        xlabel='Cluster',
        ylabel=f'Proportion of {label_col}',
        label_fontsize=18,
        tick_fontsize=18,
        figsize=(17, 10)
    )

In [ ]:
cluster_data

In [ ]:
def plot_images_per_cluster(cluster_data, image_dir, n_images, label_columns):
    """
    Plot images per cluster with their labels and cluster id.
    
    Args:
        cluster_data: DataFrame with cluster_label, image_paths, and label columns
        image_dir: directory containing the images (not used directly, paths are in cluster_data)
        n_images: maximum number of images to plot per cluster
        label_columns: list of label column names to display
    """
    unique_clusters = sorted(set(cluster_data.cluster_label))
    
    for cluster_id in unique_clusters:
        cluster_subset = cluster_data[cluster_data.cluster_label == cluster_id].reset_index(drop=True)
        
        # Plot as many images as available up to n_images
        n_to_plot = min(n_images, len(cluster_subset))
        
        print(f"\nCluster {cluster_id} — showing {n_to_plot} of {len(cluster_subset)} images")
        
        # Plot in rows of 4
        n_cols = 4
        n_rows = int(np.ceil(n_to_plot / n_cols))
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))
        axes = np.array(axes).reshape(-1)  # Flatten to 1D for easy indexing
        
        for i in range(n_to_plot):
            image_path = cluster_subset.image_paths[i]
            image = Image.open(image_path).convert('L')
            image = image.resize(target_size, Image.Resampling.LANCZOS)
            
            # Build title from label values
            label_str = '\n'.join([f"{col}: {cluster_subset[col][i]}" for col in label_columns])
            title = f"Cluster {cluster_id}\n{os.path.basename(image_path)}\n{label_str}"
            
            axes[i].imshow(image, cmap='gray')
            axes[i].set_title(title, fontsize=7)
            axes[i].axis('off')
        
        # Hide unused axes
        for i in range(n_to_plot, len(axes)):
            axes[i].axis('off')
        
        plt.tight_layout()
        plt.show()


# User defined: number of images to plot per cluster
N_IMAGES_TO_PLOT = 90

plot_images_per_cluster(cluster_data, image_dir, N_IMAGES_TO_PLOT, label_columns)